# DPQA compilation pipeline

The selected surface-code circuit is defined in circuit.py. DPQA routes its CZ interactions, then source-circuit single-qubit gates are inserted into the generated instructions and animation.

This experiment requires a separately obtained, locally modified OLSQ-DPQA checkout. The external compiler and its modifications are not distributed in this repository; see the root README for the required extensions.

## Select the source circuit

In [ ]:
from circuit import build_17_qubit_surface_code_circuit

source_circuit = build_17_qubit_surface_code_circuit()
print("Source operations:", source_circuit.count_ops())

In [ ]:
source_circuit.draw(output="mpl", fold=-1)

## Extract the DPQA interactions

DPQA solves the CZ interaction graph. The ordered source operations are passed separately so single-qubit gates can be placed before the appropriate Rydberg layers, including a final layer. The selected circuit has Hadamards between CZ blocks, so its CZ gates must not be declared globally commutable.

In [ ]:
from pathlib import Path

if not Path("dpqa_compiler/DPQA/single_qubit.py").is_file():
    raise RuntimeError(
        "This notebook needs a separate locally extended DPQA checkout; "
        "see the OLSQ-DPQA section in README.md."
    )

from dpqa_compiler.DPQA.single_qubit import extract_circuit_operations

circuit_operations, two_qubit_gates = extract_circuit_operations(source_circuit)

print(f"DPQA interactions: {len(two_qubit_gates)}")
print(f"Single-qubit gates: {sum(op['type'] == '1q' for op in circuit_operations)}")
two_qubit_gates

## Compile and export instructions

The architecture tuple gives interaction-site columns and rows, then AOD columns and rows. An optimal ratio of 1 requests a full optimal solve. The MP4 requires ffmpeg.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

dpqa_dir = Path("dpqa_compiler/DPQA").resolve()
dpqa_python = dpqa_dir / "venv" / "bin" / "python"
if not dpqa_python.exists():
    dpqa_python = Path(sys.executable)

result_name = "surface_code_d3"
command = [
    str(dpqa_python),
    str(dpqa_dir / "run.py"),
    "--graph-json", json.dumps(two_qubit_gates),
    "--circuit-ops-json", json.dumps(circuit_operations),
    "--num-qubits", str(source_circuit.num_qubits),
    "--name", result_name,
    "--architecture", "5", "5", "20", "20",
    "--optimal-ratio", "1",
    "--animate",
]

env = os.environ.copy()
env["MPLCONFIGDIR"] = str(dpqa_dir / "results" / ".matplotlib")
env["MPLBACKEND"] = "Agg"
completed = subprocess.run(
    command,
    cwd=dpqa_dir,
    env=env,
    capture_output=True,
    text=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr, file=sys.stderr)
completed.check_returncode()

results_dir = dpqa_dir / "results"
smt_path = results_dir / "smt" / f"{result_name}.json"
instructions_path = results_dir / "code" / f"{result_name}_instructions.json"
animation_path = results_dir / "animations" / f"{result_name}.mp4"

with instructions_path.open() as file:
    dpqa_instructions = json.load(file)

print(f"Generated {len(dpqa_instructions['steps'])} DPQA instruction steps.")
print(f"SMT layout: {smt_path}")
print(f"Instructions: {instructions_path}")
print(f"Visualization: {animation_path}")

## Notes

Change the builder in the first code cell to select another circuit from circuit.py. DPQA still optimizes only CZ routing: inserted single-qubit layers preserve logical order but their hardware timing and addressability are not optimized. Barriers are ignored; measurements and other unsupported operations are rejected.